# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AQ') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.1-flash-lite'

API key looks good so far


In [4]:
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have Gemini Flash figure out which links are relevant

### Use a call to Gemini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [9]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'linkedin profile',
   'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [12]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.1-flash-lite
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'professional profile',
   'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'company website',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [13]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


{'links': [{'type': 'company page',
   'url': 'https://huggingface.co/huggingface'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [14]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [16]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash
Updated
about 16 hours ago
•
6
•
1.35k
openbmb/MiniCPM5-2B
Updated
about 15 hours ago
•
42.3k
•
1.11k
XHToken/Spark-X2.5-4B
Updated
8 days ago
•
15.9k
•
1.06k
ISTA-DASLab/Qwen3.8-27B-GSQ-RCO-GGUF
Updated
9 days ago
•
615k
•
795
Qwen/Qwen3.8-27

In [24]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [18]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [19]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 5 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ndeepseek-ai/DeepSeek-V4.1-Flash\nUpdated\nabout 16 hours ago\n•\n6\n•\n1.35k\nopenbmb/MiniCPM5-2B\nUpdated\nabout 15

In [20]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [21]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 7 relevant links


# Welcome to Hugging Face: The AI Community Building the Future

Hugging Face is the leading collaboration platform for the global machine learning community. We provide the infrastructure, tools, and social ecosystem required for researchers, developers, and organizations to create, discover, and build the future of AI together.

### Our Mission
At Hugging Face, we believe that the next generation of artificial intelligence should be built in the open. Our platform serves as the central hub where the world comes to collaborate on machine learning models, datasets, and interactive applications. By democratizing access to AI tools, we are enabling innovation across the globe.

### For Customers: Enterprise-Grade AI
Whether you are a startup or a global enterprise, Hugging Face offers scalable solutions to integrate AI into your workflow:
*   **The Hugging Face Hub:** Access millions of open-source models and datasets to accelerate your development cycles.
*   **Enterprise Support & Inference:** We provide robust inference endpoints, storage buckets, and dedicated support to ensure your AI applications are production-ready and performant.
*   **Custom Solutions:** From private infrastructure to enterprise-level integrations, we help businesses leverage the power of the open-source ecosystem securely and efficiently.

### Our Culture & Community
Hugging Face is defined by its vibrant, open, and inquisitive community. Our culture is built on:
*   **Radical Openness:** We believe in sharing knowledge. Through our blog, daily research papers, and active Discord and forum channels, we foster a culture of transparency and continuous learning.
*   **Collaboration:** We are not just a tool; we are a meeting place. Developers, data scientists, and AI researchers gather here to trade ideas, fine-tune models, and solve complex problems collectively.
*   **Innovation:** We stay at the cutting edge of the industry—from hosting the latest breakthroughs in agentic AI and video generation to tracking the global compute landscape.

### Build Your Career with Us
We are looking for passionate individuals who want to define the future of technology. Working at Hugging Face means:
*   **Impact:** Your work will directly influence how millions of developers and researchers build AI.
*   **Growth:** Engage with the brightest minds in machine learning and stay ahead of industry trends.
*   **Purpose:** Contribute to an ecosystem that values open-source accessibility and the ethical, collaborative development of AI.

**Join the AI community building the future.**
Explore our platform, browse our models, or join our team to start your journey with us.

Visit us at [huggingface.co](https://huggingface.co)

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [22]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model="gemini-3.1-flash-lite",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [23]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 8 relevant links


# Hugging Face: The AI Community Building the Future

Hugging Face is the world’s leading collaborative platform for machine learning. Often described as the "GitHub of AI," it provides the essential infrastructure where researchers, developers, and organizations discover, create, and deploy artificial intelligence models, datasets, and applications.

## A Global Hub for Innovation
Hugging Face serves as a central ecosystem for the open-source AI community. Our platform hosts over 2 million models and 500,000 datasets, providing a digital home for the most cutting-edge advancements in machine learning. Whether you are exploring generative video, language processing, or complex agentic workflows, Hugging Face is the collaborative engine powering the next generation of AI.

## Key Features for the ML Ecosystem
*   **Models & Datasets:** Access a vast, searchable library of open-source models and datasets to accelerate your projects.
*   **Spaces:** A powerful environment to host, share, and demo AI applications directly in your browser.
*   **HuggingChat:** Explore conversational AI through our open-source chat interface.
*   **Enterprise Solutions:** We provide robust support for teams and enterprises, including Inference Endpoints, specialized storage buckets, and professional support to help companies build and scale AI solutions securely.

## Our Culture
Hugging Face is built on the pillars of **openness, collaboration, and community-driven progress.** We believe the future of AI is collaborative. Our culture thrives on transparency, as evidenced by our active research contributions, daily paper discussions, and open communication channels on Discord and our community forums. We are a team dedicated to democratizing AI by providing the tools and knowledge necessary for everyone to participate in the "AI revolution."

## Join Our Mission
We are always looking for passionate individuals to help us build the infrastructure that defines the future of machine learning. If you are a software engineer, researcher, or community builder who cares about open source and the ethical advancement of AI, we invite you to be part of our journey.

## Who We Serve
*   **Developers:** Build faster with access to the world’s largest collection of pre-trained models.
*   **Researchers:** Publish your findings, host your datasets, and collaborate with a global network of peers.
*   **Enterprises:** Leverage our Enterprise Support and scalable infrastructure to integrate, deploy, and monitor AI models in production environments.

**Discover what you can build at [huggingface.co](https://huggingface.co).**

In [25]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.1-flash-lite
Found 6 relevant links


# Meet Hugging Face: The AI Greenhouse (No Actual Hugging Required)

### Welcome to the Neighborhood, Neighbor
If you’ve ever wondered where all those genius-level robots, image-generating wizards, and code-writing sidekicks go to hang out, the answer is Hugging Face. We aren’t just a company; we are the central nervous system of the global AI community. 

We’re the place where over two million models, half a million datasets, and a million+ AI apps live, breathe, and occasionally try to pass a Turing test.

### What We Actually Do
Think of us as the "GitHub for AI," but with more emojis and significantly less existential dread. 
*   **The Hub:** A sprawling digital metropolis where developers build, share, and obsess over machine learning models. 
*   **Spaces:** Our playground. It’s where you can deploy your latest AI creation, host a demo, or watch a computer try to turn your face into a pumpkin—all in real-time.
*   **The Ecosystem:** Whether you’re a solo dev with a dream or an Enterprise titan with a budget, we provide the infrastructure, inference endpoints, and storage buckets to make sure your AI doesn't just work, but thrives.

### The Culture: Chaos, Code, and Collaboration
We are powered by the open-source spirit. Our offices (and our Discord) are a vibrant melting pot of researchers, dreamers, and people who spend way too much time thinking about "Agentic RL." 

We believe that AI is too important to be kept in a dark room under lock and key. We build out in the open, we share our research, and we definitely, 100% support the occasional "DeepSeek Moment" to keep things interesting.

### Why You Should Join the Fun
*   **For Customers:** Stop reinventing the wheel. Tap into a repository of 2M+ models and scale your AI operations with enterprise-grade support.
*   **For Investors:** We aren't just riding the wave; we *are* the ocean. We are the backbone of the global AI revolution, and our growth is as exponential as the compute power we manage.
*   **For Recruits:** Do you want to build the future? Do you enjoy working with the smartest people in the room while maintaining a healthy sense of humor? We provide the tools, the community, and the snacks. (Okay, we focus on the tools, but the community is pretty great too.)

### The "Must-Haves"
Our favorite color is that iconic, optimistic yellow (#FFD21E). Our mascot is a literal emoji. Our goal is to make machine learning accessible to every human being on the planet—or at least the ones with a decent internet connection.

**Join the movement. Come build the future with us.**

*Warning: Joining Hugging Face may lead to significant increases in your ability to explain complex neural networks at dinner parties, which may or may not decrease your number of dinner party invitations.*

**Hugging Face: The AI community building the future. (And yes, we accept high-fives.)**

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>